# 2026-11-25 Baselines on FOODS, clean version
Same work as 24 Nov, in one line: set up → look → yardstick → baselines → one experiment (differencing) → what we learned.
Rules for this notebook: MASE only (plus one bias line), one global number per model, every score on raw units, every chart on the same dates.

## Steps
- [x] Set up: load, build the 30 series, constants
- [x] Level: store × dept
- [x] Hold out the last 28 days
- [x] Look at the 30 series
- [x] Yardstick: MASE, 1-year scale
- [x] Baselines on raw data: naive, seasonal naive 7, seasonal naive 30
- [x] Experiment: stationarity, lag-7 difference
- [x] Baselines on the differenced data, scored in units
- [x] Why differencing didn't help
- [x] Bottomline + standup chart
- [x] Appendix: per-series detail, for debugging only

## Step 1: set up

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings
import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import adfuller, kpss
from m5food import data, plots
plots.set_style()

In [ ]:
# Constants: change them here only
HORIZON = 28                 # test days
M = 7                        # season length: one week
SCALE_DAYS = 365             # MASE yardstick: last year of training
PLOT_START = '2016-02-01'    # every chart starts here, so raw and forecast charts show the same dates

In [ ]:
# 30 daily series: one per store x food department (units sold per day)
sales = data.add_readable_names(data.load_sales_foods())
cal = data.load_calendar()
days = data.day_columns(sales)
idx = pd.DatetimeIndex(cal.set_index('d').loc[days, 'date'].values, name='date')
series = sales.groupby(['store', 'dept'])[days].sum().T.set_axis(idx)

stores = sorted(series.columns.get_level_values(0).unique())
depts = sorted(series.columns.get_level_values(1).unique())
STATE_COLOR = {'California': plots.PALETTE[0], 'Texas': plots.PALETTE[1], 'Wisconsin': plots.PALETTE[2]}
print(series.shape, '| first day', idx[0].date(), '| last day', idx[-1].date())

## Step 2: the level
*(Hyndman ch. 11)*
- Store and dept cross each other, so the data is grouped, not a clean tree.
- Items: ~14,000 series, mostly zeros. Too noisy to forecast. Not our level.
- Total: smooth, but hides the store and dept differences.
- So: **store × dept, 30 series.** Forecast these, add up to store and total (bottom-up from here).

**Ask Chiranjit:** at what level does the client actually decide?

## Step 3: hold out the last 28 days

In [ ]:
train, test = series.iloc[:-HORIZON], series.iloc[-HORIZON:]
print(f'train: {train.index[0].date()} to {train.index[-1].date()} ({len(train)} days)')
print(f'test:  {test.index[0].date()} to {test.index[-1].date()} ({len(test)} days)')

## Step 4: look at the 30 series

In [ ]:
def grid(lines, title, subtitle, zero_line=False):
    """30 panels (rows = stores, columns = depts), all from PLOT_START to the last test day.
    lines = {legend name: (DataFrame with the 30 columns, line style)}. color='state' = colour by state.
    The test period is shaded grey."""
    fig = make_subplots(rows=len(stores), cols=len(depts), shared_xaxes=True,
                        column_titles=[f'<b>{d}</b>' for d in depts], row_titles=stores,
                        vertical_spacing=0.012, horizontal_spacing=0.04)
    for r, store in enumerate(stores, start=1):
        for c, dept in enumerate(depts, start=1):
            for name, (df, style) in lines.items():
                y = df[(store, dept)].loc[PLOT_START:]
                line = dict(style)
                if line.get('color') == 'state':
                    line['color'] = STATE_COLOR[store.split(' store ')[0]]
                fig.add_trace(go.Scatter(x=y.index, y=y, name=name, line=line, mode='lines',
                                         showlegend=(r == 1 and c == 1 and len(lines) > 1),
                                         hovertemplate=f'{store}, {dept}<br>%{{x|%a %d %b}}: %{{y:,.0f}}<extra>{name}</extra>'),
                              row=r, col=c)
            if zero_line:
                fig.add_hline(y=0, line=dict(color='#888', width=1), row=r, col=c)
    fig.add_vrect(x0=test.index[0], x1=test.index[-1], fillcolor='#e6e6e6', opacity=0.6, layer='below',
                  line_width=0, row='all', col='all')
    fig.update_yaxes(nticks=3, tickfont=dict(size=9))
    fig.update_annotations(selector=dict(textangle=90), textangle=0, xanchor='left', font=dict(size=10))
    fig.update_layout(title=f'<b>{title}</b><br><span style="font-size:13px;color:#444">{subtitle}</span>',
                      template='simple_white', height=1500, margin=dict(t=110, r=150),
                      legend=dict(orientation='h', y=1.03, x=1, xanchor='right'))
    fig.show()

grid({'units a day': (series, dict(color='state', width=1.2))},
     'The 30 series we forecast: 10 stores × 3 food departments',
     f'Units sold per day, {pd.Timestamp(PLOT_START):%d %b} to {test.index[-1]:%d %b %Y}. Grey = the 28 test days. '
     'Colour = state (blue California, orange Texas, green Wisconsin). Each panel has its own scale.')

**What I see**
- Every panel has the same saw-tooth: up at the weekend, down mid-week.
- Food dept 3 is the big one. Dept 1 and 2 are a few hundred a day.
- Some months sit higher than others (SNAP days early in the month).

## Step 5: the yardstick, MASE
*(Hyndman 5.8)*
- No unit, so a big store and a small store can be compared.
- 1.0 = as good as "same weekday last week" was over the last training year. Below 1 = better.
- Scale from the last year only. Store jumps make a longer window unfair (appendix A3).
- One number per model: average MASE over the 30 series.
- Plus one bias line: too high or too low overall.

In [ ]:
recent = train.iloc[-SCALE_DAYS:]
scale = (recent - recent.shift(M)).abs().mean()          # per series: average miss of "same weekday last week"

def mase(forecast):
    """MASE per series. forecast is in units, same shape as test."""
    return (test - forecast).abs().mean() / scale

def bias(forecast):
    """% over (+) or under (-) the actual, all 30 series and 28 days together."""
    return (forecast.to_numpy().sum() / test.to_numpy().sum() - 1) * 100

def report(forecasts, title):
    """One bar per model (average MASE over 30), then one bias line."""
    score = {name: mase(f).mean() for name, f in forecasts.items()}
    best = min(score, key=score.get)
    fig = go.Figure(go.Bar(x=list(score), y=list(score.values()), text=[f'{v:.2f}' for v in score.values()],
                           textposition='outside', hovertemplate='%{x}<br>MASE %{y:.2f}<extra></extra>',
                           marker_color=[plots.PALETTE[0] if n == best else '#b8c0c6' for n in score]))
    fig.add_hline(y=1, line=dict(color='#333', dash='dot'), annotation_text='1 = same as last week, last year',
                  annotation_position='top left')
    fig.update_layout(title=f'<b>{title}</b><br><span style="font-size:13px;color:#444">Average MASE over the 30 series. '
                            'Lower is better.</span>', yaxis=dict(title='average MASE', range=[0, max(score.values()) * 1.25]),
                      template='simple_white', height=420, margin=dict(t=100))
    fig.show()
    print('Bias:', ' | '.join(f'{n} {bias(f):+.1f}%' for n, f in forecasts.items()))
    return score

## Step 6: baselines on raw data
- **Naive:** the last known day (Sun 24 Apr) for all 28 days.
- **Seasonal naive 7:** the last known week (18 to 24 Apr), repeated 4 times.
- **Seasonal naive 30:** the same date 30 days earlier.

In [ ]:
naive = pd.DataFrame([train.iloc[-1]] * HORIZON, index=test.index)
snaive7 = pd.DataFrame(np.tile(train.iloc[-M:].to_numpy(), (HORIZON // M, 1)), index=test.index, columns=train.columns)
snaive30 = series.shift(30).loc[test.index]           # 30 > 28, so it only uses training days

raw = {'naive': naive, 'seasonal naive 7': snaive7, 'seasonal naive 30': snaive30}
grid({'actual': (series, dict(color='black', width=1.2)),
      'naive': (naive, dict(color=plots.PALETTE[1], width=1.5, dash='dash')),
      'seasonal naive 7': (snaive7, dict(color=plots.PALETTE[0], width=1.5, dash='dot')),
      'seasonal naive 30': (snaive30, dict(color=plots.PALETTE[3], width=1.5, dash='dashdot'))},
     'Three baselines vs actual, 30 series',
     f'Same dates as Step 4. Grey = the 28 test days. Each panel has its own scale.')

In [ ]:
raw_score = report(raw, 'Seasonal naive 7 is the baseline to beat')

**What I see**
- Seasonal naive 7 wins: **1.06**. Keeping the weekday is worth more than anything else here.
- Naive is a flat line (1.60). It can't follow the week.
- Seasonal naive 30 is the worst (1.71). 30 days = 4 weeks + 2 days, so every weekday lands 2 days off.
- But seasonal naive 7 is **9% too low** overall. The week it copies was quiet, May had SNAP.

## Step 7: experiment, are the series stationary?

In [ ]:
def stationarity(df, title):
    """KPSS and ADF, each on its own, for the 30 series. Green = stationary, red = not.
    KPSS starts from "stationary": p < 0.05 -> red.  ADF starts from "not stationary": p < 0.05 -> green.
    KPSS only reports p between 0.01 and 0.10."""
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        p = {'KPSS': pd.Series({c: kpss(df[c], regression='c', nlags='auto')[1] for c in df}),
             'ADF': pd.Series({c: adfuller(df[c], autolag='AIC')[1] for c in df})}
    red = {'KPSS': p['KPSS'] < 0.05, 'ADF': p['ADF'] >= 0.05}
    fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.03,
                        subplot_titles=[f'<b>{t}</b>: {30 - red[t].sum()} stationary, {red[t].sum()} not' for t in p])
    for j, t in enumerate(p, start=1):
        z = red[t].unstack().loc[stores, depts].astype(float)
        text = np.where(z == 1, 'not', 'stationary') + '<br>p ' + p[t].unstack().loc[stores, depts].map('{:.3f}'.format)
        fig.add_trace(go.Heatmap(z=z.to_numpy(), x=depts, y=stores, text=text, texttemplate='%{text}',
                                 textfont=dict(size=11), zmin=0, zmax=1, showscale=False, xgap=3, ygap=3,
                                 colorscale=[[0, '#8fd19e'], [1, '#e8a3a3']],
                                 hovertemplate='%{y}, %{x}<br>%{text}<extra></extra>'), row=1, col=j)
    fig.update_yaxes(autorange='reversed')
    fig.update_layout(title=f'<b>{title}</b>', template='simple_white', height=600, margin=dict(t=100, l=150))
    fig.show()

stationarity(train, 'Raw training data: KPSS and ADF, each on its own')

**What I see**
- KPSS: all 30 red. The level moves in every series.
- ADF: 12 pass, 18 fail. Only the flatter series pass.
- Hyndman uses KPSS. So: not stationary. Try a seasonal difference (lag 7).

In [ ]:
# Seasonal difference, lag 7: today minus the same weekday last week. Then both tests again.
train_d = (train - train.shift(M)).dropna()
stationarity(train_d, 'After a lag-7 seasonal difference: KPSS and ADF, each on its own')

In [ ]:
grid({'change': ((series - series.shift(M)), dict(color='state', width=1.1))},
     'The 30 series after a lag-7 seasonal difference',
     'Units today minus units on the same weekday last week. Grey line = no change. Same dates as Step 4.',
     zero_line=True)

**What I see**
- One lag-7 difference and all 30 pass both tests.
- The lines bounce around zero. No drift.
- Stationary now. Next question: does it forecast better?

## Step 8: baselines on the differenced data
- Forecast the weekly change, not the units.
- Add it back: week 1 builds on the last real week, week 2 on forecast week 1, and so on.
- Score in units with the same MASE. No scoring on the differenced data.

In [ ]:
def undo_weekly_diff(change):
    """Change forecast -> units. Each week = the week before + the forecast change."""
    prev, weeks = train.iloc[-M:].to_numpy(), []
    for k in range(HORIZON // M):
        prev = prev + change.iloc[M * k: M * (k + 1)].to_numpy()
        weeks.append(prev)
    return pd.DataFrame(np.vstack(weeks), index=test.index, columns=train.columns)

change = series - series.shift(M)
on_diff = {
    'naive': undo_weekly_diff(pd.DataFrame([train_d.iloc[-1]] * HORIZON, index=test.index)),
    'seasonal naive 7': undo_weekly_diff(pd.DataFrame(np.tile(train_d.iloc[-M:].to_numpy(), (HORIZON // M, 1)),
                                                      index=test.index, columns=train.columns)),
    'seasonal naive 30': undo_weekly_diff(change.shift(30).loc[test.index]),
}
print('Forecast days below zero units:', {n: int((f < 0).sum().sum()) for n, f in on_diff.items()})

In [ ]:
# Standup chart: each model on raw data vs on weekly-differenced data
diff_score = {n: mase(f).mean() for n, f in on_diff.items()}
groups = ['Naive<br>(last day)', 'Seasonal naive 7<br>(same weekday last week)', 'Seasonal naive 30<br>(same date last month)']
fig = go.Figure([
    go.Bar(name='on raw data', x=groups, y=list(raw_score.values()), marker_color=plots.PALETTE[0],
           text=[f'{v:.2f}' for v in raw_score.values()], textposition='outside',
           hovertemplate='%{x}, raw<br>MASE %{y:.2f}<extra></extra>'),
    go.Bar(name='on weekly-differenced data', x=groups, y=list(diff_score.values()), marker_color=plots.PALETTE[1],
           marker_pattern_shape='/', text=[f'{v:.2f}' for v in diff_score.values()], textposition='outside',
           hovertemplate='%{x}, differenced<br>MASE %{y:.2f}<extra></extra>')])
fig.add_hline(y=1, line=dict(color='#333', dash='dot'), annotation_text='1 = same as last week, last year',
              annotation_position='top left')
fig.update_layout(title='<b>Seasonal naive 7 on raw data is still the baseline to beat</b><br>'
                        '<span style="font-size:13px;color:#444">30 store × food dept series, test 25 Apr to 22 May 2016. '
                        'Average MASE, scored in units. Lower is better.</span>',
                  barmode='group', bargap=0.3, template='simple_white', height=480, margin=dict(t=110),
                  yaxis=dict(title='average MASE', range=[0, max(list(raw_score.values()) + list(diff_score.values())) * 1.2]),
                  legend=dict(orientation='h', y=-0.2, x=0.5, xanchor='center'))
fig.show()
print('Bias on differenced:', ' | '.join(f'{n} {bias(f):+.1f}%' for n, f in on_diff.items()))

## Step 9: why differencing didn't help
- The model copies last week's *change* and adds it again every week.
- One odd week → counted 4 times by week 4. Errors pile up.
- Naive 1.60 → 2.43, seasonal naive 7 1.06 → 2.52. Both ~23% too low.
- Forecast the change as 0 → you get seasonal naive 7 back. Which is already the best.
- So stationary ≠ better forecast. What matters is what you forecast for the change.
- Seasonal naive 30 is the one that improves: adding back onto the last real week fixes its 2-day weekday slip (1.71 → 1.55). Still far behind 1.06.

## Step 10: bottomline
- **Level:** store × food dept, 30 daily series. Add up to store and total.
- **Test:** last 28 days, 25 Apr to 22 May 2016.
- **Metric:** MASE, 1-year scale. One number per model.
- **Scores:** naive 1.60 · seasonal naive 7 **1.06** · seasonal naive 30 1.71.
- **Baseline to beat:** seasonal naive 7. It keeps the weekday, but runs 9% too low (SNAP weeks).
- **Differencing:** makes the data stationary (30/30), makes these baselines worse.
- **Next:** a model that keeps the weekly shape, follows the level, and knows SNAP and events.

## Appendix (debugging only)
### A1: MASE per series, every model

In [ ]:
per_series = pd.DataFrame({**{f'{n} (raw)': mase(f) for n, f in raw.items()},
                           **{f'{n} (diff)': mase(f) for n, f in on_diff.items()}})
per_series = per_series.sort_values('seasonal naive 7 (raw)', ascending=False)
labels = [f'{s} · {d}' for s, d in per_series.index]
fig = go.Figure(go.Heatmap(z=per_series.to_numpy(), x=per_series.columns, y=labels, zmin=0.5, zmax=2.5,
                           colorscale=[[0, '#8fd19e'], [0.25, '#f7f7f7'], [1, '#e8a3a3']],
                           text=per_series.round(2).to_numpy(), texttemplate='%{text}', textfont=dict(size=10),
                           colorbar=dict(title='MASE'), xgap=2, ygap=2,
                           hovertemplate='%{y}<br>%{x}: MASE %{z:.2f}<extra></extra>'))
fig.update_yaxes(autorange='reversed')
fig.update_layout(title='<b>MASE per series</b><br><span style="font-size:13px;color:#444">Green below 1, red above. '
                        'Sorted by seasonal naive 7, worst on top.</span>',
                  template='simple_white', height=950, margin=dict(t=100, l=260))
fig.show()

### A2: the 3 worst series for seasonal naive 7

In [ ]:
worst = mase(snaive7).nlargest(3).index
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=[f'{s} · {d}  (MASE {mase(snaive7)[(s, d)]:.2f})' for s, d in worst])
for r, col in enumerate(worst, start=1):
    y = series[col].loc[PLOT_START:]
    fig.add_trace(go.Scatter(x=y.index, y=y, name='actual', line=dict(color='black', width=1.2), showlegend=r == 1), row=r, col=1)
    fig.add_trace(go.Scatter(x=snaive7.index, y=snaive7[col], name='seasonal naive 7', showlegend=r == 1,
                             line=dict(color=plots.PALETTE[0], width=1.8, dash='dot')), row=r, col=1)
fig.add_vrect(x0=test.index[0], x1=test.index[-1], fillcolor='#e6e6e6', opacity=0.6, layer='below', line_width=0,
              row='all', col='all')
fig.update_layout(title='<b>Where seasonal naive 7 misses most</b>', template='simple_white', height=750,
                  margin=dict(t=100), legend=dict(orientation='h', y=1.06, x=1, xanchor='right'))
fig.show()

### A3: why a 1-year scale (seasonal naive 7 only)

In [ ]:
scale_5y = (train - train.shift(M)).abs().mean()
err = (test - snaive7).abs().mean()
check = pd.DataFrame({'5-year scale': err / scale_5y, '1-year scale': err / scale}).sort_values('1-year scale')
labels = [f'{s} · {d}' for s, d in check.index]
fig = go.Figure()
for lab, a, b in zip(labels, check['5-year scale'], check['1-year scale']):
    fig.add_trace(go.Scatter(x=[a, b], y=[lab, lab], mode='lines', line=dict(color='#ddd', width=3),
                             showlegend=False, hoverinfo='skip'))
fig.add_trace(go.Scatter(x=check['5-year scale'], y=labels, mode='markers', name='5-year scale',
                         marker=dict(color='#999', size=9, symbol='circle-open', line=dict(width=2))))
fig.add_trace(go.Scatter(x=check['1-year scale'], y=labels, mode='markers', name='1-year scale',
                         marker=dict(color=plots.PALETTE[0], size=9)))
fig.add_vline(x=1, line=dict(color='#333', dash='dot'))
fig.update_layout(title='<b>Seasonal naive 7: MASE with a 5-year vs a 1-year scale</b><br>'
                        f'<span style="font-size:13px;color:#444">Same forecast, only the scale changes. Average '
                        f'{check["5-year scale"].mean():.2f} → {check["1-year scale"].mean():.2f}. '
                        'Long lines = series that jumped.</span>',
                  xaxis_title='MASE', template='simple_white', height=820, margin=dict(t=110, l=260),
                  legend=dict(orientation='h', y=-0.08, x=0.5, xanchor='center'))
fig.show()